In [16]:
# --- Load Line Following Model (unchanged) ---
import torchvision
import torch
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
model.load_state_dict(torch.load('best_steering_model_xy.pth'))
device = torch.device('cuda')
model = model.to(device)
model = model.eval().half()

# --- Load Sign Detection Model ---
sign_model = torchvision.models.resnet18(pretrained=False)
sign_model.fc = torch.nn.Linear(512, 3)
sign_model.load_state_dict(torch.load('best_model_resnet18.pth'))
sign_model = sign_model.to(device).eval().half()

CLASS_NAMES = ['Left', 'NoSign', 'Right']

# --- Shared preprocessing (same normalization) ---
import torchvision.transforms as transforms
import PIL.Image
import numpy as np
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std  = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

# --- Line Following Control System (from document) ---
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import cv2, time
import torch.nn.functional as F
import queue, threading
from PID import PID

# Parameters
forward_speed_default = 0.12
steering_bias_default = 0.00
IN_W, IN_H = 224, 224

camera = Camera.instance(width=IN_W, height=IN_H)
robot = Robot()

# UI widgets
view_widget = widgets.Image(format='jpeg', width=IN_W, height=IN_H)
start_button = widgets.Button(description="START", button_style='success',
                               layout=widgets.Layout(width='120px', height='40px'))
stop_button = widgets.Button(description="STOP", button_style='danger',
                              layout=widgets.Layout(width='120px', height='40px'))

kp_slider = widgets.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.001, description='Kp')
kd_slider = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.001, description='Kd')
speed_slider = widgets.FloatSlider(value=forward_speed_default, min=0.0, max=0.6, step=0.005, description='Speed')
bias_slider = widgets.FloatSlider(value=steering_bias_default, min=-0.5, max=0.5, step=0.005, description='Bias')

x_readout = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.001, description='x', readout=True)
y_readout = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.001, description='y', readout=True)
ang_readout = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=0.1, description='angle (deg)', readout=True)

# *** ADD: Sign detection display widget ***
sign_label = widgets.HTML("<h2 style='text-align: center; color: gray;'>No Sign</h2>")

display(
    widgets.VBox([
        view_widget,
        sign_label,  # *** Sign detection display ***
        widgets.HBox([start_button, stop_button]),
        widgets.HBox([kp_slider, kd_slider]),
        widgets.HBox([speed_slider, bias_slider]),
        widgets.HBox([x_readout, y_readout, ang_readout]),
    ])
)

# PID setup
steer_pid = PID(Kp=kp_slider.value, Ki=0.0, Kd=kd_slider.value, setpoint=0.0)

def reset_pid(*_):
    steer_pid.reset()
    steer_pid.set_Kp(kp_slider.value)
    steer_pid.set_Kd(kd_slider.value)
    steer_pid.set_setpoint(0.0)

kp_slider.observe(reset_pid, names='value')
kd_slider.observe(reset_pid, names='value')

# Control state
running = False

def on_start(_):
    global running
    running = True

def on_stop(_):
    global running
    running = False
    robot.stop()

start_button.on_click(on_start)
stop_button.on_click(on_stop)

def clamp(v, lo, hi):
    return max(lo, min(hi, float(v)))

def xy_to_px(x, y, W, H):
    px = int((x * 0.5 + 0.5) * (W - 1))
    py = int((y * 0.5 + 0.5) * (H - 1))
    return max(0, min(W-1, px)), max(0, min(H-1, py))

frame_q = queue.Queue(maxsize=1)

def on_frame(change):
    frame = change['new']
    if frame is None:
        return
    if frame_q.full():
        try:
            frame_q.get_nowait()
        except queue.Empty:
            pass
    frame_q.put_nowait(frame)

@torch.no_grad()
def process_frame(frame_bgr):
    global running, steer_pid

    if frame_bgr is None:
        return
    
    vis = frame_bgr.copy()
    H, W = vis.shape[:2]
    
    # Preprocess once for both models
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    input_tensor = preprocess(rgb)
    
    # *** LINE FOLLOWING (unchanged) ***
    xy = model(input_tensor).float().squeeze().tolist()
    x, y = float(xy[0]), float(xy[1])
    
    x_readout.value = x
    y_readout.value = y
    ang_readout.value = float(np.clip(x * 90.0, -180.0, 180.0))
    
    steer_pid.set_Kp(kp_slider.value)
    steer_pid.set_Kd(kd_slider.value)
    steer_pid.set_setpoint(0.0)
    
    error = steer_pid.update(x)
    
    fwd = speed_slider.value
    left, right = clamp(fwd - error, 0.0, 1.0), clamp(fwd + error, 0.0, 1.0)
    
    # *** SIGN DETECTION (parallel) ***
    sign_output = sign_model(input_tensor)
    p = F.softmax(sign_output, dim=1).squeeze().float().cpu().numpy()
    
    left_prob = p[0]
    nosign_prob = p[1]
    right_prob = p[2]
    
    if left_prob > 0.9:
        result = "Left"
        color = "blue"
    elif right_prob > 0.9:
        result = "Right"
        color = "green"
    elif nosign_prob > 0.7:
        result = "No Sign"
        color = "gray"
    else:
        result = "Uncertain"
        color = "orange"
    
    sign_label.value = f"<h2 style='text-align: center; color: {color};'>{result}</h2>"
    
    # Draw line following overlays
    px, py = xy_to_px(x, y, W, H)
    cv2.circle(vis, (px, py), 5, (0, 0, 255), -1)
    cv2.putText(vis, f"Kp={kp_slider.value:.2f} Kd={kd_slider.value:.3f}",
                (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40,255,40), 1, cv2.LINE_AA)
    cv2.putText(vis, f"Error={error:.3f}",
                (8, H - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
    
    # *** ADD: Display sign on video ***
    cv2.putText(vis, f"Sign: {result}",
                (8, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1, cv2.LINE_AA)
    
    if running:
        robot.set_motors(left, right)
    else:
        robot.stop()

    return bgr8_to_jpeg(vis)

def processing_loop():
    while True:
        frame_bgr = frame_q.get()
        view_widget.value = process_frame(frame_bgr)

camera.observe(on_frame, names='value')
threading.Thread(target=processing_loop, daemon=True).start()

In [14]:
camera.stop()

In [15]:
robot.stop()